In [4]:
! pip install langchain newsdataapi credentials langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.25
    Uninstalling langchain-core-0.3.25:
      Successfully uninstalled langchain-core-0.3.25
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.12
    Uninstalling langchain-0.3.12:
      Successfully uninstalled langchain-0.3.12


In [8]:
%pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.1/109.1 kB 9.3 MB/s eta 0:00:00


In [ ]:
from newsdataapi import NewsDataApiClient
from langchain.agents import initialize_agent, Tool
from langchain.agents import AgentType
from langchain_groq import ChatGroq
from time import sleep
from google.colab import userdata


In [ ]:
from newsdataapi import NewsDataApiClient
import random
import requests

class NewsExtractor:
    """
    A class for extracting the latest news based on a search query.
    """

    def __init__(self, NEWSDATA_KEY: str, search_query: str = None):
        """
        Initializes the NewsExtractor instance.

        Args:
            NEWSDATA_KEY (str): The API key for the NewsDataApiClient.
            search_query (str, optional): The variable to be searched. Defaults to None.
        """
        # self.api_client = NewsDataApiClient(apikey=NEWSDATA_KEY)
        self.search_query = search_query

    def get_news(self, query: str = None) -> list:
        """
        Retrieves the latest news based on a search query.

        Args:
            query (str, optional): The variable to be searched. Overrides the instance's search_query.

        Returns:
            list: A list of strings containing the latest news.
        """
        
        search_query = self.search_query

        try:
            
            if search_query:
                print("*****",search_query)
                url = f"https://newsapi.org/v2/everything?q={search_query}&apiKey={NEWSDATA_KEY}"

                response = requests.get(url)
                response = response.json()
                print(response)
               
            else:
              response = {}
                

            # Extract news content
            news = [article['title'] for article in response.get('articles', [])]

            if not news:
                return ["No news found for the given query."]

            # Return the news list
            return random.choice(news)

        except Exception as e:
            # Handle exceptions gracefully
            return [f"Error fetching news: {str(e)}"]



In [ ]:
def news_summarizer(NEWSDATA_KEY:str, GROQ_API_KEY:str, limit:str="175 characters", total_news:int=5, sleep_seconds:int=20, search_query:str=None):
    '''
    Function that retrieves the latest news and generates short summaries for the news using an AI model.

    Args:
        NEWSDATA_KEY (str): The API key for the NewsDataApiClient.
        OPENAI_KEY (str): The API key for the ChatOpenAI model.
        limit (str): The maximum character limit for each news summary. Defaults to "175 characters".
        total_news (int): The number of news articles to summarize. Defaults to 5.
        sleep_seconds (int): The number of seconds to wait between each news retrieval and summarization. Defaults to 20.
        search_query (str, optional): The variable to be searched. Defaults to None.

    Returns:
        list: A list of responses containing the generated news summaries.
    '''


    llm = ChatGroq(
    model="mixtral-8x7b-32768",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)
    NE = NewsExtractor(NEWSDATA_KEY, search_query)

    newssummary = Tool(
        name='newssummary',
        func=NE.get_news,
        description='get news based on the search topic',
    )  
    tools = [newssummary]  
    agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True, h)  # Initializing the agent with the provided tools and ChatOpenAI model
    
    responses = []  

    prompt = f"""
    Your task is to get the latest news and then generate a short summary for this news \
    Summarize the news
    in at most {limit}. Try to be as short as possible. Never ever cross the {limit} limit for summarized news.
    """  

    for _ in range(total_news):
        response = agent.run(prompt)  
        responses.append(response)  
        sleep(sleep_seconds)  

    return responses 

In [ ]:
# GROQ_API_KEY= userdata.get('GROQ_API_KEY')
# NEWSDATA_KEY = userdata.get('NEWSDATA_KEY')


import os
GROQ_API_KEY= os.environ.get('GROQ_API_KEY')
NEWSDATA_KEY = os.environ.get('NEWSDATA_KEY')

# Set your Groq API key as an environment variable
os.environ["GROQ_API_KEY"] =GROQ_API_KEY
os.environ["NEWSDATA_KEY"] = NEWSDATA_KEY



search_query = input("Enter search query")

response = news_summarizer(NEWSDATA_KEY, GROQ_API_KEY, limit="500 characters", total_news = 2,sleep_seconds = 20,search_query = search_query)




> Entering new AgentExecutor chain...
To answer this question, I need to get the latest news and then summarize it in a concise manner while ensuring that the summary does not exceed 500 characters.

Action: newssummary
Action Input: "latest news"
***** bitcoin
{'status': 'ok', 'totalResults': 10364, 'articles': [{'source': {'id': None, 'name': 'Gizmodo.com'}, 'author': 'Matthew Gault', 'title': 'Bitcoin ATM Security Breach Compromised Social Security Numbers and Government IDs', 'description': "Byte Federal operates 1,200 Bitcoin ATMs in the U.S. A data breach comprised 58,000 customer's information.", 'url': 'https://gizmodo.com/bitcoin-atm-security-breach-comprised-customers-price-of-bitcoin-2000537845', 'urlToImage': 'https://gizmodo.com/app/uploads/2024/12/BitcoinATM.jpg', 'publishedAt': '2024-12-12T15:30:41Z', 'content': 'A massive data breach hit Bitcoin ATM company Byte Federal, compromising user information including their social security number, transaction history, and eve

In [78]:
response

['Bitcoin whales are selling their holdings, causing a dip in the crypto market. The reason for this sell-off is not clear, but it may be due to profit-taking or anticipation of a market downturn.',
 "Trump's SEC pick and Bitcoin's $100K milestone are noteworthy events in the crypto community, potentially signaling a more favorable regulatory environment and a major price achievement for the digital currency."]